# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 mass spectrometry dataset of human extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and prepare the Dataset object for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load and inspect dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}\n{'='*len(metadata.name)}\n{metadata.description}\n")

## 2. Data Overview
List the available record sets and their fields by `@id`. This will help navigate the structure of the dataset. Make sure to reference all elements by their `@id`.

In [ ]:
# List record sets and their fields, referencing all entities by their `@id`
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print(f"Record sets found: {record_set_ids}\n")

field_ids_by_record_set = {}
for rs in metadata.to_json().get('recordSet', []):
    rs_id = rs['@id']
    print(f"Record Set: {rs_id}\n  Name: {rs.get('name', '')}\n  Description: {rs.get('description', '')}")
    field_ids = []
    for field in rs.get('field', []):
        if isinstance(field, dict):
            field_id = field.get('@id')
        else:
            field_id = field
        field_ids.append(field_id)
    field_ids_by_record_set[rs_id] = field_ids
    print(f"  Field @ids: {field_ids}\n")

## 3. Data Extraction
Load all record sets using their `@id` and collect them as DataFrames ready for analysis.

In [ ]:
# Load all record sets as DataFrames using their `@id`
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for {record_set_id}: shape = {df.shape}")
    else:
        print(f"\nNo records found for {record_set_id}.")

# For demonstration, pick the main protein abundance table (usually such a set exists)
if dataframes:
    main_rs = list(dataframes.keys())[0]  # Just as an example
    print(f"\nColumns in {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    dataframes[main_rs].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
As an example, process one record set using only its field/column `@id` for referencing fields.
<br/>
**Select a numeric field to filter and normalize (for demonstration, we use a field with 'Abundance' or 'MW' in its name). Group by the accession or a sequence identifier if available.**

In [ ]:
# Identify candidate numeric field and group field by @id using field_ids from section 2
import numpy as np

target_df = None
for record_set_id, df in dataframes.items():
    # Try to pick a DataFrame with numeric columns called 'MW', 'Abundance', 'Count', etc.
    numeric_candidates = [col for col in df.columns if any(key in col for key in ['MW', 'Abundance', 'Count', 'coverage', 'mass', 'weight'])]
    if numeric_candidates:
        main_rs_id = record_set_id
        main_numeric_field = numeric_candidates[0]
        print(f"Using record set {record_set_id}, numeric field: {main_numeric_field}")
        target_df = df
        break
if target_df is None:
    print("No suitable numeric field found in any record set.")
else:
    # Find group field (prefer accession, id, gene, etc)
    possible_groups = [col for col in target_df.columns if any(key in col for key in ['accession', 'Access', 'gene', 'description', 'type', 'class'])]
    group_field = possible_groups[0] if possible_groups else None
    # Filter: remove non-numeric or NA values
    vals = pd.to_numeric(target_df[main_numeric_field], errors='coerce')
    threshold = np.nanpercentile(vals, 90)  # filter for top 10%
    filtered_df = target_df[vals > threshold].copy()
    print(f"Filtered records where {main_numeric_field} > {threshold:.2f} (top 10%).")
    normalized_col = main_numeric_field + "_normalized"
    filtered_vals = pd.to_numeric(filtered_df[main_numeric_field], errors='coerce')
    filtered_df[normalized_col] = (filtered_vals - filtered_vals.mean()) / filtered_vals.std()
    print(filtered_df[[main_numeric_field, normalized_col]].head())
    if group_field:
        grouped_df = filtered_df.groupby(group_field, as_index=False)[normalized_col].mean()
        print(f"\nGrouped mean normalized {main_numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and, if grouped, visualize group means.

In [ ]:
import matplotlib.pyplot as plt

if target_df is not None:
    # Visualize original and normalized numeric field distribution
    fig, axs = plt.subplots(1, 2, figsize=(12,4))
    pd.to_numeric(target_df[main_numeric_field], errors='coerce').hist(ax=axs[0], bins=30, color='lightblue', edgecolor='black')
    axs[0].set_title(f"Distribution of {main_numeric_field}")

    filtered_df[normalized_col].hist(ax=axs[1], bins=30, color='lightgreen', edgecolor='black')
    axs[1].set_title(f"Normalized {main_numeric_field} (top 10%)")
    plt.show()

    if group_field:
        grouped_df_sorted = grouped_df.sort_values(by=normalized_col, ascending=False).head(20)
        grouped_df_sorted.plot(kind='bar', x=group_field, y=normalized_col, legend=False, figsize=(10,5))
        plt.ylabel(f"Mean normalized {main_numeric_field}")
        plt.title(f"Top 20 groups by mean normalized {main_numeric_field}")
        plt.show()

## 6. Conclusion
- This notebook demonstrated how to discover and reference all entities by their `@id` in the FAIR^2 dataset via the `mlcroissant` API.
- We loaded and inspected the schema, explored available record sets and their fields, extracted the main abundance table, performed basic filtering and normalization, grouped by protein `@id`, and visualized distributions as a foundation for downstream proteomics analytics.

For more advanced processing or bioinformatics pipelines, refer to the specific field `@id`s surfaced in your own schema exploration using this workflow.